In [1]:
import pandas as pd
import os
import json


"""
场景假设：
1. 用户在前端页面上，通过pathway的搜索栏，选择了一个pathway，前端返回一个json文件
2. 后端解析json文件，根据pathway名字，从Mt_KEGG_filter表中找到对应的基因的list
3. 根据基因list，从Mt_Exp表或者Mt_Exp_fname表找到对应的基因的表达谱
"""


def json_load(path):
    """
    Description: 读取json文件,转为字典
    """
    with open(path) as f:
        dict = json.load(f)

    return dict


def pathway_name_json(json_path):
    """
    Description: 根据json文件内容，提取pathway的名字
    """
    pathway_name = json_load(json_path)['pathway']

    return pathway_name


def pathway_gene_list_json(json_path, kegg_filter_path):
    """
    Description: 根据pathway名称提取相应基因list
    Arguments:
        json_path: json文件的路径
        kegg_filter_path: kegg_filter表的路径
    Returns:
        gene_list: Pathway对应的基因的list
    """
    df_kegg_filter = pd.read_csv(kegg_filter_path, sep='\t')

    # 根据json文件内容，提取pathway的名字
    pathway_name = pathway_name_json(json_path)

    # 根据KEGG_filter表的表头，找到对应的基因的list
    df = df_kegg_filter[df_kegg_filter['Pathway'] == pathway_name]
    gene_list = df['Gene'].tolist()

    return gene_list


def gene_exp_df_json(json_path, kegg_filter_path, exp_path):
    """
    Description: 根据json文件解析出的pathway的名字，找到对应的基因的list，然后根据df_exp，返回对应基因的表达量
    Arguments:
        json_path: json文件的路径
        kegg_filter_path: kegg_filter表的路径
        exp_fname_path: exp表的路径
    Returns:
        json_str: 表达谱的json字符串
    """
    df_exp = pd.read_csv(exp_path, sep='\t')

    # 根据pathway的名字，找到对应的pathway中基因的list
    gene_list = pathway_gene_list_json(json_path, kegg_filter_path)

    # 根据基因的list，找到对应的基因的表达量
    df = df_exp[df_exp['Gene id'].isin(gene_list)]

    json_str = df.to_json(orient='records')
    
    return json_str


def gene_exp_fname_df_json(json_path, kegg_filter_path, exp_fname_path):
    """
    Description: 根据json文件解析出的pathway的名字，找到对应的基因的list，然后根据df_exp_fname，返回对应基因的表达量
    Arguments:
        json_path: json文件的路径
        kegg_filter_path: kegg_filter表的路径
        exp_fname_path: exp_fname表的路径
    Returns:
        json_str: 表达谱的json字符串
    """
    df_exp_fname = pd.read_csv(exp_fname_path, sep='\t')
    
    # 根据pathway的名字，找到对应的pathway中基因的list
    gene_list = pathway_gene_list_json(json_path, kegg_filter_path)

    # 根据基因的list，找到对应的基因的表达量
    df = df_exp_fname[df_exp_fname['Gene id'].isin(gene_list)]

    json_str = df.to_json(orient='records')

    # with open(json_output_path,'w') as f:
    #     f.write(json_str)
    
    return json_str

In [2]:
pathway_gene_list_json('./pathway.json','./Mt_KEGG_filter.tsv')
# gene_exp_df_json('./pathway.json','./Mt_KEGG_filter.tsv','./Mt_Exp.tsv')
# gene_exp_fname_df_json('./pathway.json','./Mt_KEGG_filter.tsv','./Mt_Exp_fname.tsv')

['MYCTH_2300427',
 'MYCTH_2311835',
 'MYCTH_2309135',
 'MYCTH_2294864',
 'MYCTH_2296558',
 'MYCTH_2306649',
 'MYCTH_2046198',
 'MYCTH_2307926',
 'MYCTH_2296237',
 'MYCTH_2303752',
 'MYCTH_2304608',
 'MYCTH_2306082',
 'MYCTH_2311550',
 'MYCTH_2305149',
 'MYCTH_2309414',
 'MYCTH_2311423',
 'MYCTH_2305350',
 'MYCTH_2312321',
 'MYCTH_2305097',
 'MYCTH_2309135',
 'MYCTH_2305051',
 'MYCTH_2305828']